# 14 — Historical-fill cache (Experiment 2 cloud handling)

Advisor's point 3 applied to clouds: a masked pixel is filled from the **same site's own clean
history** — same parcel, same land class, same position — instead of the chip mean (as shipped)
or a generated image (genfill, dropped). Rule: per-pixel median of the site's P01–P08 chips where
the pixel is finite in ≥ 3 periods; else median over ≥ 1 period; else chip mean. History is
P01–P08 for *every* period (P09/P10 targets included), so no target chip informs its own fill
(`panel_histfill.py`). Chips are then re-encoded with the frozen TerraMind tokenizers exactly as
in notebook 01 → `latents_biweekly_histfill.npz`.

Sentinel-1 chips are ≥ 99 % valid: the only nodata is a permanent one-pixel edge column at
six sites (never observed, so it stays chip-mean filled) and 71 px in one chip; Sentinel-1 latents
are therefore essentially the chip-mean cache. The fill acts on Sentinel-2.


In [1]:
import sys, json
sys.path.insert(0, ".")
import panel_lib as pl
print("CUDA_VISIBLE_DEVICES =", pl.pick_gpu())
import numpy as np, pandas as pd
import panel_histfill as ph
pd.set_option("display.width", 220)
idx = pl.build_panel_index()
tokb = pl.load_tokenizers()
lat, counts = ph.build_cache(idx, tokb)
manifest = ph.write_cache(lat, counts)
print(json.dumps(manifest, indent=1))


CUDA_VISIBLE_DEVICES = 0


/data/wang/junh/githubs/latent-synthetic-control/Satellite/notebooks/ts_SCM_ASCM/panel_histfill.py:37: RuntimeWarning: All-NaN slice encountered
  med = np.nanmedian(stack, axis=0)                     # NaN where count == 0


sentinel1: 64/1147
sentinel1: 128/1147
sentinel1: 192/1147


sentinel1: 256/1147
sentinel1: 320/1147
sentinel1: 384/1147


sentinel1: 448/1147
sentinel1: 512/1147
sentinel1: 576/1147


...

sentinel2: 832/1175
sentinel2: 896/1175


sentinel2: 960/1175
sentinel2: 1024/1175


sentinel2: 1088/1175
sentinel2: 1152/1175
sentinel2: 1175/1175


encode determinism gate sentinel2: max|diff| = 0.00e+00


{
 "date": "2026-08-28",
 "base": "biweekly chips (same 2322 images as latents_biweekly.npz)",
 "fill": "per-pixel median of the same site's P01-P08 chips where the pixel is finite in >= 3 periods; else median over >= 1 period; else chip mean (Tok.prep fill_nan). History = P01-P08 for EVERY period.",
 "encoder": "terramind_v1_tokenizer_s1grd / s2l2a, batch 64, determinism gate",
 "n_latents": 2322,
 "fill_totals": {
  "n_masked": 4426424,
  "n_filled_median": 4245937,
  "n_filled_any": 160308,
  "n_chipmean_fallback": 20179
 }
}


## Fill accounting

In [2]:
c2 = counts.query("sensor == 'sentinel2'")
print("Sentinel-2 chips with any masked pixel:", int((c2.n_masked > 0).sum()), "of", len(c2))
print("fully masked chips (10201 px):", int((c2.n_masked == 10201).sum()))
tot = c2[["n_masked", "n_filled_median", "n_filled_any", "n_chipmean_fallback"]].sum()
print("pixels: masked", int(tot.n_masked), "| filled from >=3-period median", int(tot.n_filled_median),
      "| from 1-2 periods", int(tot.n_filled_any), "| chip-mean fallback", int(tot.n_chipmean_fallback))
print("\nper-site history depth (max periods a pixel is observed in P01-P08), Sentinel-2:")
print(c2.groupby("site_id").history_periods.first().describe().round(1).to_string())
print("\nthe 10 treated sites, P09/P10 (the validation targets):")
print(c2.query("site_id.str.startswith('treatment') and seq in [9, 10]")
      [["site_id", "period_id", "n_masked", "n_filled_median", "n_filled_any", "n_chipmean_fallback"]].to_string(index=False))
c1 = counts.query("sensor == 'sentinel1'"); print("\nSentinel-1 masked pixels total:", int(c1.n_masked.sum()))


Sentinel-2 chips with any masked pixel: 797 of 1175
fully masked chips (10201 px): 145
pixels: masked 4416213 | filled from >=3-period median 4245866 | from 1-2 periods 160308 | chip-mean fallback 10039

per-site history depth (max periods a pixel is observed in P01-P08), Sentinel-2:
count    60.0
mean      6.5
std       0.8
min       5.0
25%       6.0
50%       6.0
75%       7.0
max       8.0

the 10 treated sites, P09/P10 (the validation targets):
       site_id  period_id  n_masked  n_filled_median  n_filled_any  n_chipmean_fallback
treatment_0001 before_P09      6571             6571             0                    0
treatment_0001 before_P10         0                0             0                    0
treatment_0002 before_P09      2749             1443          1306                    0
treatment_0002 before_P10         0                0             0                    0
treatment_0003 before_P09         0                0             0                    0
treatment_0003 bef

## Sanity: Sentinel-1 latents unchanged; Sentinel-2 latents change only where pixels were filled

In [3]:
orig = pl.load_latents(pl.LATD / "latents_biweekly.npz")
d1 = max(float(np.abs(orig[k] - lat[k]).max()) for k in lat if k[1] == "sentinel1")
print("Sentinel-1 max |histfill - chipmean| over all latents:", d1)
rows = []
for k, v in lat.items():
    if k[1] != "sentinel2": continue
    nm = counts.query("site_id == @k[0] and sensor == 'sentinel2' and period_id == @k[2]").n_masked.iloc[0]
    rows.append({"masked_px": nm, "latent_change": float(np.abs(orig[k] - v).mean())})
r = pd.DataFrame(rows); r["bin"] = pd.cut(r.masked_px, [-1, 0, 1000, 5000, 10200, 10201], labels=["0", "1-1000", "1001-5000", "5001-10200", "all"])
print(r.groupby("bin", observed=True).latent_change.agg(["count", "mean", "max"]).round(4).to_string())


Sentinel-1 max |histfill - chipmean| over all latents: 0.5714285373687744


            count    mean     max
bin                              
0             378  0.0000  0.0000
1-1000        172  0.0182  0.1224
1001-5000     191  0.0949  0.2292
5001-10200    289  0.1952  0.4473
all           145  0.3541  0.4136


## Reading

1. **Sentinel-2: 797 of 1,175 chips had masked pixels, 145 were fully masked.** Of 4.42 M masked
   pixels, 96.1 % were filled from a ≥ 3-period median of the site's own P01–P08 chips, 3.6 % from a
   1–2-period median, and 0.2 % (10,039 px) fell back to the chip mean. Every site has ≥ 5 clean
   observations of most pixels in P01–P08 (median 6).
2. **The validation targets.** At P09, 8 of the 10 treated sites had masked pixels (site 0010:
   9,581 of 10,201 px; 0001: 6,571; 0005: 6,215; 0008: 3,577) — all filled from history. At P10 only
   site 0004's permanent 101-px edge column is masked (chip-mean fallback, as in every period). So
   the P10 test is a clean-target test under both caches, and P09 is where the fill matters.
3. **Latent change scales with the fill**: mean |Δ latent| 0.02 for chips with ≤ 1,000 masked px,
   0.10 for 1,001–5,000, 0.20 for 5,001–10,200 and 0.35 for fully masked chips (which the
   chip-mean cache had encoded as constant images). Unmasked chips are bit-identical.
4. **Sentinel-1**: 10,211 masked pixels in total — a permanent nodata edge column (101 px) at
   sites counterfactual_0003_02, 0004_01, 0005_01, 0010_02 and treatment_0004 (chip-mean fallback,
   never observed) and 71 px in counterfactual_0009_04 after_P05 (history-filled); one latent
   differs from the chip-mean cache by one FSQ level, the rest are identical.
